In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import StandardScaler

from aare.constants import LOC_BERN, TIME, TEMP
from aare.preparation import (
    resample,
    remove_faulty_periods_aare_temp,
    remove_outliers_aare_temp,
    interpolate_aare_temp,
)
from aare.remote_existenz_store import RemoteExistenzStore
from aare.utils import to_ts, between

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
store = RemoteExistenzStore()

In [ ]:
ANYTIME = "0"  # to be used as period start when querying influx. starting at 0 just returns all the data.
START_TIME = "2022-01-01"

In [ ]:
df = store.query_hydro(START_TIME, LOC_BERN)
o_df = df.copy()
df

In [ ]:
df = resample(df)
df

In [ ]:
df = remove_faulty_periods_aare_temp(df)
df

In [ ]:
df = remove_outliers_aare_temp(df)
df

In [ ]:
df = interpolate_aare_temp(df)
df

In [ ]:
px.scatter(df, TIME, TEMP, color="filled").update_traces(marker={"size": 3})

In [ ]:
ts = to_ts(df.drop("filled", axis=1))
ts

In [ ]:
gaps = ts.gaps().sort_values("gap_size", ascending=False)
gaps

In [ ]:
X = df[TEMP].values
X.shape

In [ ]:
X = X.reshape(-1, 1)
X.shape

In [ ]:
scaler = StandardScaler()

In [ ]:
X = scaler.fit_transform(X)
X.shape

In [ ]:
imputation_horizon = 7 * 24

In [ ]:
from pypots.data import sliding_window

# no overlap
# X = sliding_window(X, imputation_horizon, imputation_horizon)

# every half-week is in two windows (once at start, once at end)
X = sliding_window(X, imputation_horizon, imputation_horizon // 2)
X.shape

In [ ]:
X_orig = X.copy()

In [ ]:
from pygrinder import seq_missing

X = seq_missing(X, 0.05, seq_len=25)
X = seq_missing(X, 0.05, seq_len=45)
X.shape

In [ ]:
dataset = dict(X=X)

In [ ]:
from pypots.imputation import SAITS

saits = SAITS(
    n_steps=imputation_horizon,
    n_features=1,
    n_layers=2,
    d_model=256,
    d_ffn=128,
    n_heads=4,
    d_k=64,
    d_v=64,
    dropout=0.1,
    epochs=100,
    saving_path="tensorboard",  # set the path for saving tensorboard logging file and model checkpoint
    model_saving_strategy="best",  # only save the model with the best validation performance
    device="cuda",
)

In [ ]:
saits.fit(dataset)

In [ ]:
imputation = saits.impute(dataset)

In [ ]:
from pypots.utils.metrics import calc_mae

mae = calc_mae(imputation, np.nan_to_num(X_orig), np.isnan(X) ^ np.isnan(X_orig))
mae

In [ ]:
gap = gaps.iloc[0]
gap

In [ ]:
df[
    between(
        df,
        pd.to_datetime(gap["gap_start"], utc=True),
        pd.to_datetime(gap["gap_end"], utc=True),
    )
]

In [ ]:
gap_start_i = df[df[TIME] == pd.to_datetime(gap["gap_start"], utc=True)].index[0]
gap_start_i

In [ ]:
non_gap = imputation_horizon - gap["gap_size"]
pre_gap = non_gap // 2
input_start_i = gap_start_i - pre_gap
input_end_i = input_start_i + imputation_horizon - 1

input_start_i, input_end_i

In [ ]:
input_chunk = df.loc[input_start_i:input_end_i].copy()
assert len(input_chunk) == imputation_horizon
input_chunk

In [ ]:
px.scatter(input_chunk, TIME, TEMP)

In [ ]:
# load the best model we got. No idea why this one was that good, probably just luck with the seed.
# This was just to toy around, of course for real training need better tracking
# and some hyperparameter tuning. Also better repro :)
# saits.load("tensorboard/20241229_T161204/SAITS.pypots")

In [ ]:
input = input_chunk[TEMP].values.reshape(-1, 1)  # steps / features
input = scaler.transform(input)
input = input.reshape(1, -1, 1)  # samples / steps / features
output = saits.impute(dict(X=input))
output = output.reshape(-1, 1)  # steps / features
output = scaler.inverse_transform(output)
output = output.reshape(-1)  # steps
output

In [ ]:
input_chunk["imputed"] = output

In [ ]:
px.scatter(input_chunk, TIME, ["imputed", TEMP])